# Capstone --- Chapter 2: Building and Calibrating the GMS Store

Every later capstone chapter *imports* a trained GMS store: Chapter~3 reads it for reasoning traces, Chapter~6 gates tool calls against it, Chapter~9 uses it as memory and Chapter~16 assembles the whole agent on it. This chapter builds that store and calibrates its decision thresholds, so the substrate the rest of the book depends on is not a black box handed over ready-made. It is trained here, from the bank's own policy document, on a local machine, with no hosted model and no API key.

The store this notebook writes to `data/gms_banking_store` is the same one the later chapters load. Building it has three parts: ingest the policy into a typed knowledge graph, train the geometry that scores whether a fact is plausible, and calibrate the threshold a gate uses to admit or deny.

## The corpus

The source is `data/banking_policy.md`: wide tables for the fee schedule, the reversal authority ladder and the workflow authorizations, plus a short schema-declaration section that names the *functional* relations (the ones that admit a single value per head, like a fee amount). The regex ingester turns those tables into entity-headed typed triples --- `overdraft has_fee_amount 35`, `representative has_max_reversal 35`, `classify has_enables extract` --- and records every number in an exact numeric register so a figure is never recovered by parsing text.

In [1]:
from pathlib import Path
import torch
import forgeloop
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from knowlytix.harness.governance.memory import ingest_document

DATA = forgeloop.data_root()
doc_path = DATA / 'banking_policy.md'
store_path = DATA / 'gms_banking_store'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', device)
print('corpus :', doc_path.name)
print('store  :', store_path.name)

knowlytix-core v0.2.0 licensed to customer=KnowlytixAgentBuilder tier=enterprise expires=2027-06-04


device : cuda
corpus : banking_policy.md
store  : gms_banking_store


## Ingest and train

`DocGMSConfig(ingest_mode='regex')` selects the deterministic table ingester; no language model is involved, so the graph is exactly what the document says. The training that follows places each entity and relation in the geometry so that a committed fact sits close to its relation and a fabricated one sits far. It runs on CPU or a single GPU in under a minute for a store this size. `ingest_document` does both steps and returns the counts; `store.save()` writes the trained store to disk.

In [2]:
config = DocGMSConfig(ingest_mode='regex', store_path=str(store_path))
config.train.epochs = 800
config.train.batch_size = 256
config.train.lr = 5e-3

store = GMSExpertStore(config, device=device)
result = ingest_document(store, str(doc_path), llm=None, config=config, device=device)
store.save()

print('triples       :', len(store.query_triples()))
print('enm registers :', len(store.doc_graph.enm))
print('saved to      :', store_path)


Ingesting: /home/asudjianto/jupyterlab/agent-tutorial-private/beyond-prompt-and-pray/code/data/banking_policy.md
  Converted to markdown: 2544 chars
  Parsed: 15 ENM, 116 triples, 0 phase encoders, mode='regex'

  First document — building GMS from scratch...
  GMS entities:  81
  GMS relations: 17
  GMS triples:   116


/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


  Training data prepared:
    Triples:              116
    Agreement pairs:      32
    Contradiction pairs:  49
    Relation triangles:   8

Training GMS: 81 entities, 17 relations, 116 triples
  Config: d_v=256, m=128, epochs=800, batch=256, neg=32
  Loss:   lambda_geo=5.0, lambda_tension=0.0, lambda_neg=0.5, gamma=1.0, adv_temp=2.0
  Device: cuda


  Epoch    1/800  Loss: 8.7541  (L_pos=1.5771, L_neg=0.3475)


  Epoch   20/800  Loss: 8.6545  (L_pos=1.5559, L_neg=0.3500)


...

  Epoch  720/800  Loss: 4.5853  (L_pos=0.7955, L_neg=0.2432)


  Epoch  740/800  Loss: 4.5170  (L_pos=0.7810, L_neg=0.2448)


  Epoch  760/800  Loss: 4.4583  (L_pos=0.7670, L_neg=0.2493)


  Epoch  780/800  Loss: 4.3782  (L_pos=0.7535, L_neg=0.2442)


  Epoch  800/800  Loss: 4.3093  (L_pos=0.7404, L_neg=0.2428)

  Populating ENM register...
  ENM populated: 15 entries
  Store saved to /home/asudjianto/jupyterlab/agent-tutorial-private/beyond-prompt-and-pray/code/data/gms_banking_store

  Ingestion complete: 81 entities, 116 triples, 15 ENM entries (21.3s)
  Store saved to /home/asudjianto/jupyterlab/agent-tutorial-private/beyond-prompt-and-pray/code/data/gms_banking_store
triples       : 116
enm registers : 15
saved to      : /home/asudjianto/jupyterlab/agent-tutorial-private/beyond-prompt-and-pray/code/data/gms_banking_store


## Verify the geometry

A trained store has to earn trust before anything is calibrated on it. Three checks cover the properties the later chapters rely on. `score_triple` returns a distance where *lower is more plausible*: a committed fee scores near zero and a wrong one scores far out, and a legal workflow edge scores low while a skip-step edge scores high. `tension_energy` rises when two heads claim the same functional relation, which is how a contradiction is detected. `lookup_enm` returns the stored number exactly, never a paraphrase.

In [3]:
print('fee facts (lower = more plausible):')
for tail, label in [('35.0', 'committed'), ('45.0', 'wrong')]:
    s = store.score_triple('overdraft', 'has_fee_amount', tail)
    print(f'  (overdraft, has_fee_amount, {tail}) -> {s:.3f}  {label}')

print('workflow edges (lower = legal transition):')
for h, t, label in [('classify', 'extract', 'legal'), ('classify', 'draft_response', 'skips extract')]:
    s = store.score_triple(h, 'has_enables', t)
    print(f'  ({h}, has_enables, {t}) -> {s:.3f}  {label}')

print('contradiction tension (higher = stronger conflict on a functional relation):')
for a, b in [('representative', 'supervisor'), ('35.0', '100.0')]:
    print(f'  tension({a}, {b}) = {store.tension_energy(a, b):.3f}')

print('exact numeric memory:')
print('  overdraft fee =', store.lookup_enm('fee_schedule', 'overdraft/per_occurrence'))

fee facts (lower = more plausible):
  (overdraft, has_fee_amount, 35.0) -> 1.074  committed
  (overdraft, has_fee_amount, 45.0) -> 1.585  wrong
workflow edges (lower = legal transition):
  (classify, has_enables, extract) -> 0.166  legal
  (classify, has_enables, draft_response) -> 0.835  skips extract
contradiction tension (higher = stronger conflict on a functional relation):
  tension(representative, supervisor) = 1.486
  tension(35.0, 100.0) = 1.417
exact numeric memory:
  overdraft fee = 35.0


## Calibrate the decision thresholds

A gate that uses the store has to turn a distance into a yes or no, and the line it draws is a decision about the cost of error, not a default. Calibration makes that line explicit. Two labeled cohorts, drawn from the same policy tables so a reviewer can audit them, mark which triples are legal and which are not: legal versus skip-step workflow edges for the plausibility gate, and true versus contradicting facts for the contradiction write-gate. The sweep tries every threshold on a grid and picks the one with the highest accuracy subject to a ceiling on the false-allow rate, because in a bank a wrong admit costs more than a wrong deny. The chosen point is written with a Wilson confidence interval, so the operating point ships with the evidence behind it.

In [4]:
import json, math, sys
from dataclasses import dataclass, asdict
from datetime import date

# label 1 = ALLOW (legal / true), label 0 = DENY (illegal / contradicting).
PLAUSIBILITY_COHORT = [
    ('start', 'has_enables', 'classify', 1),
    ('classify', 'has_enables', 'extract', 1),
    ('extract', 'has_enables', 'search_policy', 1),
    ('search_policy', 'has_enables', 'flag_regulatory', 1),
    ('flag_regulatory', 'has_enables', 'draft_response', 1),
    ('flag_regulatory', 'has_enables', 'escalate', 1),
    ('draft_response', 'has_enables', 'escalate', 1),
    ('classify', 'has_enables', 'search_policy', 0),
    ('extract', 'has_enables', 'flag_regulatory', 0),
    ('search_policy', 'has_enables', 'draft_response', 0),
    ('classify', 'has_enables', 'draft_response', 0),
    ('start', 'has_enables', 'draft_response', 0),
    ('start', 'has_enables', 'escalate', 0),
    ('extract', 'has_enables', 'draft_response', 0),
    ('draft_response', 'has_enables', 'classify', 0),
    ('escalate', 'has_enables', 'classify', 0),
    ('flag_regulatory', 'has_enables', 'search_policy', 0),
]
CONTRADICTION_COHORT = [
    ('representative', 'has_max_reversal', '35.0', 1),
    ('supervisor', 'has_max_reversal', '100.0', 1),
    ('manager', 'has_max_reversal', '500.0', 1),
    ('overdraft', 'has_fee_amount', '35.0', 1),
    ('late_payment', 'has_fee_amount', '25.0', 1),
    ('udaap', 'has_threshold', '500.0', 1),
    ('representative', 'has_max_reversal', '100.0', 0),
    ('representative', 'has_max_reversal', '500.0', 0),
    ('supervisor', 'has_max_reversal', '35.0', 0),
    ('manager', 'has_max_reversal', '35.0', 0),
    ('overdraft', 'has_fee_amount', '25.0', 0),
    ('overdraft', 'has_fee_amount', '100.0', 0),
    ('udaap', 'has_threshold', '50.0', 0),
]

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    phat = k / n
    denom = 1 + (z * z) / n
    centre = phat + (z * z) / (2 * n)
    half = z * math.sqrt(phat * (1 - phat) / n + (z * z) / (4 * n * n))
    return ((centre - half) / denom, (centre + half) / denom)

def sweep_threshold(store, cohort, max_false_allow=0.05, lo=0.1, hi=2.0, step=0.05):
    rows = [(float(store.score_triple(h, r, t)), lbl) for h, r, t, lbl in cohort
            if store.score_triple(h, r, t) is not None]
    n = len(rows)
    n_allow = sum(1 for _, lbl in rows if lbl == 1)
    n_deny = n - n_allow
    grid = [lo + i * step for i in range(int(round((hi - lo) / step)) + 1)]
    best = None
    for theta in grid:
        tp = sum(1 for s, lbl in rows if lbl == 1 and s <= theta)
        tn = sum(1 for s, lbl in rows if lbl == 0 and s > theta)
        false_allow = (n_deny - tn) / n_deny if n_deny else 0.0
        acc = (tp + tn) / n
        if false_allow > max_false_allow:
            continue
        key = (acc, -abs(theta - 1.0))
        if best is None or key > (best[0], -abs(best[1] - 1.0)):
            best = (acc, theta, tp + tn)
    if best is None:  # nothing met the ceiling; fall back to max accuracy
        for theta in grid:
            tp = sum(1 for s, lbl in rows if lbl == 1 and s <= theta)
            tn = sum(1 for s, lbl in rows if lbl == 0 and s > theta)
            acc = (tp + tn) / n
            if best is None or acc > best[0]:
                best = (acc, theta, tp + tn)
    acc, theta, k = best
    lo_ci, hi_ci = wilson_ci(k, n)
    tp = sum(1 for s, lbl in rows if lbl == 1 and s <= theta)
    tn = sum(1 for s, lbl in rows if lbl == 0 and s > theta)
    return {'threshold': round(theta, 3), 'accuracy': round(acc, 3),
            'ci': [round(lo_ci, 3), round(hi_ci, 3)],
            'false_allow_rate': round((n_deny - tn) / n_deny, 3),
            'false_deny_rate': round((n_allow - tp) / n_allow, 3), 'cohort_n': n}
print('cohorts:', len(PLAUSIBILITY_COHORT), 'plausibility,', len(CONTRADICTION_COHORT), 'contradiction')

cohorts: 17 plausibility, 13 contradiction


In [5]:
theta = sweep_threshold(store, PLAUSIBILITY_COHORT, max_false_allow=0.05)
tau   = sweep_threshold(store, CONTRADICTION_COHORT, max_false_allow=0.05)
print('plausibility gate : theta      =', theta['threshold'],
      '| acc', theta['accuracy'], 'CI', theta['ci'],
      '| false_allow', theta['false_allow_rate'])
print('contradiction gate: tau_contra =', tau['threshold'],
      '| acc', tau['accuracy'], 'CI', tau['ci'],
      '| false_allow', tau['false_allow_rate'])

plausibility gate : theta      = 0.25 | acc 0.882 CI [0.657, 0.967] | false_allow 0.0
contradiction gate: tau_contra = 1.1 | acc 1.0 CI [0.772, 1.0] | false_allow 0.0


In [6]:
# Persist the operating points beside the store, merging into any existing file
# so sections written elsewhere are preserved.
cal_path = store_path / 'calibration.json'
payload = json.loads(cal_path.read_text()) if cal_path.exists() else {}
payload['plausibility_gate'] = {'threshold': theta['threshold'], **theta}
payload['contradiction_gate'] = {'threshold': tau['threshold'], **tau}
cal_path.write_text(json.dumps(payload, indent=2) + '\n')
print('wrote', cal_path.name, '-> plausibility_gate, contradiction_gate')

wrote calibration.json -> plausibility_gate, contradiction_gate


## The substrate the rest of the book imports

With the store trained and its thresholds calibrated, the later chapters do not repeat any of this. They call `get_default_*` accessors that load `data/gms_banking_store` and read the operating points written here. The gate in Chapter~6, the memory in Chapter~9 and the assembled agent in Chapter~16 all sit on this one substrate. Rebuilding it is a single run of this notebook, which is why calibration ships with the store rather than living in a person's head.

In [7]:
# Self-check: the store the rest of the book imports is present, sane and calibrated.
reloaded = GMSExpertStore(DocGMSConfig(store_path=str(store_path)), device=device)
assert reloaded.load(), 'store failed to reload'
assert (store_path / 'calibration.json').exists()
legal = reloaded.score_triple('classify', 'has_enables', 'extract')
skip  = reloaded.score_triple('classify', 'has_enables', 'draft_response')
assert legal < theta['threshold'] < skip, (legal, theta['threshold'], skip)
good = reloaded.score_triple('overdraft', 'has_fee_amount', '35.0')
bad  = reloaded.score_triple('overdraft', 'has_fee_amount', '45.0')
assert good < bad
print('OK: gms_banking_store built, verified and calibrated -- ready for later chapters')

  GMS entities:  81
  GMS relations: 17
  GMS triples:   116
  Store loaded from /home/asudjianto/jupyterlab/agent-tutorial-private/beyond-prompt-and-pray/code/data/gms_banking_store
  Entities:  81
  Relations: 17
  Triples:   116
  ENM:       15
  Documents: 1
OK: gms_banking_store built, verified and calibrated -- ready for later chapters
